# AIWorkValue：真实实验数据怎样支持商业判断

独立重分析 Noy–Zhang (2023) 实验，并作透明的商业情景延伸。不是本项目自行实施的实验，也不是当前模型性能测试。先从项目根目录运行 `make download && make run`。

In [1]:
from pathlib import Path
import json, sys, sqlite3
ROOT = Path.cwd()
if not (ROOT / "reports/results.json").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from aiworkvalue.data import load_study
d, audit = load_study(ROOT)
r = json.loads((ROOT / "reports/results.json").read_text())
for row in audit["sample_flow"]: print(row)
print("Primary incentive arms:", int(d.primary_arm.sum()))


{'stage': 'raw_survey_records', 'n': 786, 'control': 368, 'ai_offer': 365}
{'stage': 'completed_and_not_declined', 'n': 523, 'control': 273, 'ai_offer': 250}
{'stage': 'unique_first_submission', 'n': 505, 'control': 263, 'ai_offer': 242}
{'stage': 'documented_first_task_and_duplicate_flags', 'n': 499, 'control': 260, 'ai_offer': 239}
{'stage': 'author_quality_check', 'n': 457, 'control': 237, 'ai_offer': 220}
{'stage': 'exclude_prior_graders', 'n': 453, 'control': 235, 'ai_offer': 218}
Primary incentive arms: 359


## 1. 保留分配效应的比较

实际是否使用 AI 是随机分配之后的选择，不能据此筛选处理组。固定时长组也不适合放入主要时间比较。

In [2]:
with sqlite3.connect(ROOT / "data/processed/study.sqlite") as connection:
    q = """SELECT assignment, COUNT(*), ROUND(AVG(time_pre),2),
    ROUND(AVG(time_post),2), ROUND(AVG(grade_post),3)
    FROM participants WHERE primary_arm=1 GROUP BY assignment"""
    print("assignment / N / baseline minutes / post minutes / post grade")
    for row in connection.execute(q): print(row)


assignment / N / baseline minutes / post minutes / post grade
(0, 187, 28.97, 27.04, 3.83)
(1, 172, 29.8, 16.93, 4.518)


## 2. 自己重新估计一个主模型

控制基线、职业×任务顺序及激励组。系数是分钟；负值代表更快。可用样本与原始随机入组总体要分开解释。

In [3]:
import statsmodels.formula.api as smf
primary = d[d.primary_arm].copy()
formula = "time_post ~ assignment + time_pre + C(occupation)*C(a_first) + C(incentive_arm)"
fit = smf.ols(formula, data=primary).fit(cov_type="HC3", use_t=True)
print("N:", int(fit.nobs))
print("Assignment coefficient (minutes):", round(fit.params["assignment"], 4))
print("95% interval:", fit.conf_int().loc["assignment"].round(4).tolist())
assert abs(fit.params["assignment"] - r["primary"][0]["coefficient"]) < 1e-8


N: 346
Assignment coefficient (minutes): -10.3395
95% interval: [-12.5826, -8.0964]


![Primary effects](../figures/02_primary_effects.png)

质量的单位是 1–7 分量表上的分数差，不是事实准确率。

In [4]:
for row in r["primary"]:
    print(row["name"], "N=", row["n"], "effect=", round(row["coefficient"],3),
          "CI=", [round(x,3) for x in row["ci95"]], "Holm p=", row["holm_p_value"])


primary_time N= 346 effect= -10.339 CI= [-12.583, -8.096] Holm p= 2.1972288317847033e-17
primary_quality N= 352 effect= 0.7 CI= [0.466, 0.933] Holm p= 8.987254580400753e-09


## 3. 稳健性与探索性差异

不要仅凭分组内显著与否判断两组差异；应看交互项。职业和基线写作分数也不应变成真实员工能力排行榜。

In [5]:
for row in r["robustness"]:
    print(row["name"], "N=", row["n"], "effect=", round(row["coefficient"],3))
print("Exploratory interaction tests:")
for row in r["heterogeneity"]["interaction_tests"]:
    print(row["name"], round(row["coefficient"],3), [round(x,3) for x in row["ci95"]])


unadjusted_time N= 356 effect= -10.114
time_change_design_controls N= 346 effect= -10.842
log_time_ancova N= 346 effect= -0.657
objective_page_duration N= 357 effect= -9.32
unadjusted_quality N= 352 effect= 0.688
quality_change_design_controls N= 352 effect= 0.706
quality_including_fixed_time_arm N= 444 effect= 0.721
paired_grader_fixed_effects N= 1067 effect= 0.712
Exploratory interaction tests:
time_baseline_writing_score_interaction -0.417 [-2.131, 1.297]
quality_baseline_writing_score_interaction -0.249 [-0.434, -0.063]


## 4. 亲自修改商业假设

研究估计给出历史任务中的分配效应。任务频率、可迁移比例、复核成本和席位费用是商业假设。下面修改复核时间，展示商业结论怎样反转。

In [6]:
from aiworkvalue.business import Scenario, value_case
from dataclasses import replace
saved = -r["primary"][0]["coefficient"]
for review in (1, 2, 3, 4):
    case = value_case(saved, replace(Scenario(), additional_review_minutes=review))
    print("Additional review minutes:", review,
          "monthly net capacity value:", round(case["net_capacity_value_month"],2))
print("Released capacity is not automatically cash savings.")


Additional review minutes: 1 monthly net capacity value: 1864.69
Additional review minutes: 2 monthly net capacity value: 698.03
Additional review minutes: 3 monthly net capacity value: -468.64
Additional review minutes: 4 monthly net capacity value: -1635.31
Released capacity is not automatically cash savings.


![Business sensitivity](../figures/07_business_sensitivity.png)

## 你自己的延伸

选择一种熟悉的写作工作，写清与实验任务的相同/不同之处。只改变一个有业务依据的假设，重建图表，并解释哪项新的实测最可能改变投入决策。明确标记推测，不虚构访谈、员工试验或公司收益。